In [1]:
import duckdb

con = duckdb.connect("tweedekamer.duckdb", read_only=True)
print(con.sql("SHOW TABLES").df())

Empty DataFrame
Columns: [name]
Index: []


## Feitentabel publicatie aanmaken

In [13]:
con.close()
con = duckdb.connect("tweedekamer.duckdb")

con.sql("CREATE SCHEMA IF NOT EXISTS gold")
con.sql("""
    CREATE OR REPLACE TABLE gold.fact_document_publicatie AS
    SELECT
        d.id AS document_id,
        d.soort,
        m.document_type,
        d.organisatie,
        d.datum,
        m.publicatie_datum AS publicatiedatum,
        DATE_DIFF('day', d.datum, m.publicatie_datum) AS dagen_tot_publicatie,
        v.bestandsgrootte,
        v.versienummer
    FROM silver.document d
    LEFT JOIN silver.document_versie v ON d.huidige_document_versie_id = v.id
    LEFT JOIN silver.document_publicatie_metadata m
        ON m.document_versie_id = v.id AND m.document_type = 'Metadata'
""")

print(con.sql("SELECT COUNT(*) AS aantal FROM gold.fact_document_publicatie").df())

   aantal
0  611885


## Dimentietabellen soort, type, organisatie, document aanmaken

In [14]:
con.sql("""
    CREATE OR REPLACE TABLE gold.dim_documentsoort AS
    SELECT DISTINCT soort FROM silver.document WHERE soort IS NOT NULL
""")
con.sql("""
    CREATE OR REPLACE TABLE gold.dim_documenttype AS
    SELECT DISTINCT document_type FROM gold.fact_document_publicatie WHERE document_type IS NOT NULL
""")
con.sql("""
    CREATE OR REPLACE TABLE gold.dim_organisatie AS
    SELECT DISTINCT organisatie FROM silver.document WHERE organisatie IS NOT NULL
""")
con.sql("""
    CREATE OR REPLACE TABLE gold.dim_document AS
    SELECT id AS document_id, titel FROM silver.document
""")

## Dimentietabel tijd aanmaken

In [15]:
con.sql("""
    CREATE OR REPLACE TABLE gold.dim_tijd AS
    SELECT
        CAST(d AS DATE) AS datum,
        YEAR(d) AS jaar,
        MONTH(d) AS maand,
        DAY(d) AS dag,
        QUARTER(d) AS kwartaal,
        WEEKOFYEAR(d) AS weeknummer,
        DAYNAME(d) AS dagnaam,
        ISODOW(d) AS dagnr
    FROM (
        SELECT UNNEST(GENERATE_SERIES(min_d, max_d, INTERVAL 1 DAY)) AS d
        FROM (
            SELECT
                CAST(LEAST(MIN(datum), MIN(publicatiedatum)) AS TIMESTAMP) AS min_d,
                CAST(GREATEST(MAX(datum), MAX(publicatiedatum)) AS TIMESTAMP) AS max_d
            FROM gold.fact_document_publicatie
        )
    )
""")

## Feitentabel bronvermelding aanmaken

In [16]:
con.sql("""
    CREATE OR REPLACE TABLE gold.fact_bronvermelding AS
    SELECT
        b.document_id,
        b.bron_document_id,
        d.soort AS document_soort,
        s.soort AS bron_document_soort,
        b.relatie_gewijzigd_op
    FROM silver.document_bron_document b
    LEFT JOIN silver.document d ON b.document_id = d.id
    LEFT JOIN silver.document s ON b.bron_document_id = s.id
""")

## Feitentabellen controleren

In [28]:
print(con.sql("""
    SELECT table_name, estimated_size AS rijen
    FROM duckdb_tables()
    WHERE schema_name = 'gold'
    ORDER BY table_name
""").df())

                 table_name   rijen
0              dim_document  611885
1         dim_documentsoort     106
2          dim_documenttype       1
3           dim_organisatie       3
4                  dim_tijd   13988
5       fact_bronvermelding  123887
6  fact_document_publicatie  611885


In [32]:
con.close()

In [1]:
import duckdb

con = duckdb.connect("tweedekamer.duckdb")
con.execute(open("sql/export_csv.sql", encoding="utf-8").read())
con.close()

In [2]:
import os
import pandas as pd

for naam in ["fact_document_publicatie", "fact_bronvermelding", "dim_documentsoort",
             "dim_documenttype", "dim_organisatie", "dim_tijd", "dim_document"]:
    pad = f"data/{naam}.csv"
    df = pd.read_csv(pad)
    mb = os.path.getsize(pad) / 1_000_000
    print(f"{naam}: {len(df)} rijen, {df.shape[1]} kolommen, {mb:.1f} MB")

fact_document_publicatie: 12147 rijen, 9 kolommen, 1.1 MB
fact_bronvermelding: 2471 rijen, 5 kolommen, 0.3 MB
dim_documentsoort: 106 rijen, 1 kolommen, 0.0 MB
dim_documenttype: 1 rijen, 1 kolommen, 0.0 MB
dim_organisatie: 3 rijen, 1 kolommen, 0.0 MB
dim_tijd: 13988 rijen, 8 kolommen, 0.5 MB
dim_document: 14527 rijen, 2 kolommen, 1.1 MB
